In [1]:
import kglab
import pickle
import rdflib 
import re
import torch
import json
import gc
import os
import logging
import random
import faiss
import ctranslate2

import sentencepiece as spm
import networkx as nx
import pandas as pd
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from collections import Counter, defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from rapidfuzz import process, utils, fuzz
from sparse_dot_topn import sp_matmul_topn
from community import community_louvain
from collections import Counter

from sentence_transformers import SentenceTransformer, losses, models, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.datasets import DenoisingAutoEncoderDataset
from torch.utils.data import DataLoader
from datasets import Dataset, IterableDataset
from transformers import BertLMHeadModel, pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

tqdm.pandas()

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\kglab\util.py:35: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore  # pylint: disable=E0401


In [2]:
triples = pd.read_excel("../outputs/clean_outputs/triples_coalesced.xlsx").drop("Unnamed: 0", axis=1)
triples.head()

,id,company,job title,text,triples_qwen_structured,triples_qwen_semi-structured,triples_qwen_unstructured,triples_gemma_structured,triples_gemma_semi-structured,triples_gemma_unstructured,triples_llama_structured,triples_llama_semi-structured,triples_llama_unstructured
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'netwo...","[('IT Administrator', 'require', 'technical_se...","[('IT Administrator', 'INVOLVES_TASK', 'Mainta...","[('IT Administrator', 'REQUIRES_SKILL', 'Syste...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'Docks...","[('IT Administrator', 'have', 'influence on a ..."
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'netwo...","[('IT Administrator', 'require', 'technical_se...","[('IT Administrator', 'REQUIRES_SKILL', 'IT Ad...","[('IT Administrator', 'REQUIRES_SKILL', 'Syste...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'REQUIRES_SKILL', 'Docks...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'have', 'influence on a ..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...,"[('SQE Manager', 'REQUIRES_SKILL', 'Root Cause...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality Ma...","[('SQE Manager', 'require', 'quality assurance...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'title', 'SQE Manager'), ('SQ...","[('SQE Manager', 'REQUIRES_QUALITY', 'detailed...","[('SQE Manager', 'REQUIRES_SKILL', 'DevOps Eng...","[('SQE Manager', 'require', 'English language ..."
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...,"[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'req...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'tit...","[('Customer service', 'REQUIRES_QUALITY', 'det...","[('Customer service', 'REQUIRES_SKILL', 'Techn...","[('Customer service', 'have', 'a technical sup..."
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...,"[('Retail Designer', 'REQUIRES_SKILL', 'Techni...","[('Retail Designer', 'REQUIRES_SKILL', 'Design...","[('Retail Designer', 'require', 'technical ski...","[('Retail designer', 'REQUIRES_SKILL', 'the te...","[('Retail designer', 'REQUIRES_SKILL', 'the te...",[('Retail designer with the technique in place...,[('Retail designer with the technique in place...,"[('Danish designer', 'REQUIRES_QUALITY', 'SKIL...",[('Retail designer requires technical competen...


In [3]:
df_isco = pd.read_excel("../outputs/raw_outputs/ISCO_ESCO_triples.xlsx")
df_isco.head()

,Unnamed: 0,id,ISCO cv qwen,ESCO cv qwen,ISCO gemma semi-structured,ESCO gemma semi-structured,ISCO gemma structured,ESCO gemma structured,ISCO gemma unstructured,ESCO gemma unstructured,...,ESCO llama structured,ISCO llama unstructured,ESCO llama unstructured,ISCO qwen semi-structured,ESCO qwen semi-structured,ISCO qwen structured,ESCO qwen structured,ISCO qwen unstructured,ESCO qwen unstructured,humanjobid
0,0,1,NaN,NaN,"[(""IT-administrator"", ""has_isco"", ""2522""), (""I...","[(""IT-administrator"", ""has_esco"", ""011116"")]","[(""IT-administrator"", ""has_isco"", ""2522"")]","[(""IT-administrator"", ""has_esco"", ""012179"")]","[(""IT-administrator – få indflydelse på et set...","[(""IT administration"", ""has_esco"", ""011151""), ...",...,"[('IT-administrator', 'has_esco', '000046'), ...","[('Database Administrator', 'has_isco', '2521'...",Here are the linked triples:\n\n[('IT Administ...,"[(""IT-administrator"", ""has_isco"", ""2522"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]","[(""IT-administrator"", ""has_isco"", ""2521"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]","[(""IT-administrator"", ""has_isco"", ""1330"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]",1527392
1,1,2,NaN,NaN,"[(""SQE Manager"", ""has_isco"", ""7536"")]","[(""Test Execution"", ""has_esco"", ""010569""), (""D...","[(""SQE Manager"", ""has_isco"", ""7536"")]","[(""008265"", ""has_esco"", ""apply risk management...","[(""SQE Manager"", ""has_isco"", ""1420""), (""SQE Ma...","[(""Kvalitetssikring"", ""has_esco"", ""008265""), (...",...,"After analyzing the provided triples, I have i...","[('SQE Manager', 'has_isco', '6111'), ('SQE M...",Here are the ESCO triples:\n\n [('SQE Manager'...,"[(""SQE Manager"", ""has_isco"", ""7543"")]","[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...","[(""SQE Manager"", ""has_isco"", ""7543"")]","[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...",[],"[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ...",1527395
2,2,3,NaN,NaN,"[(""Ships' Deck Officers and Pilots"", ""has_isco...","[(""012179"", ""has_esco"", ""process order forms w...",[],"[(""001160"", ""has_esco"", ""customer service"")]","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""Kundeservice / teknisk support"", ""has_esco""...",...,Here is the list of ESCO triples:\n\n [('Kunde...,"[('Kundeservice', 'har', '5329'), ('Kundeserv...",Here is the list of ESCO triples:\n\n [('Kunde...,"[(""Kundeservice / teknisk support"", ""has_isco""...","[(""customer service"", ""has_esco"", ""001160"")]","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""technical communication"", ""has_esco"", ""0065...","[(""Kundeservice / teknisk support"", ""has_isco""...","[(""customer service"", ""has_esco"", ""01160"")]",1527397
3,3,4,NaN,NaN,"[(""Retail designer"", ""REQUIRES_SKILL"", ""AutoCA...","[(""Retail designer"", ""REQUIRES_SKILL"", ""010923...",[],"[(""Retail designer"", ""has_esco"", ""001496""), (""...","[(""Retail designer med teknikken på plads"", ""h...","[(""Retail designer med teknikken på plads"", ""h...",...,"[('Retail designer med teknikken på plads', '...","[('Retail designer', 'has_isco', '7513'), ('R...",Here are the ESCO triples for the given triple...,[],"[(""use creative suite software"", ""has_esco"", ""...",[],[],[],"[(""Retail Designer"", ""has_esco"", ""006381""), (""...",1527417
4,4,5,NaN,NaN,"[(""Supporter til Caseware"", ""has_isco"", ""5312""...","[(""003500"", ""has_esco"", ""manage packaging mate...",[],[],"[(""Customer Support"", ""has_isco"", ""4419""), (""C...","[(""Communication"", ""has_esco"", ""009910""), (""Pr...",...,Here are the ESCO triples for the given subjec...,"[('Caseware', 'has_isco', 7533), ('Caseware',...",[('Supporter til Caseware – bliv kundernes su...,"[(""Supporter til Caseware"", ""has_isco"", ""5329"")]","[(""Supporter til Caseware"", ""has_esco"", ""00655...",[],"[(""apply supports for spinal adjustment"", ""has...",[],"[(""technical_support"", ""has_e

In [4]:
df_isco_cv = pd.read_excel("../outputs/raw_outputs/ISCO_ESCO_triples_cv.xlsx")
df_isco_cv[["ISCO cv qwen", "ESCO cv qwen"]].head()

,ISCO cv qwen,ESCO cv qwen
0,"[(""d296ec1752b8415b903e70f37b184854"", ""has_isc...",[]
1,"[(""2f5414d0fca34938981f8cb789e87d3c"", ""has_isc...",[]
2,"[(""34adba822a9f4e59a8f994ec95fa0e64"", ""has_isc...",[]
3,"[(""ecaae4b36d1040fb95d930ef4ce14682"", ""has_isc...","[(""give interviews to media"", ""has_esco"", ""004..."
4,"[(""b845420665d741a79b71aa42c3cb8c26"", ""has_isc...",[]


In [5]:
df_bridges = pd.read_excel("../outputs/raw_outputs/bridge_triples/bridge_table.xlsx")
df_bridges.head()

,Unnamed: 0,humanjobid,cvid,bridges_gemma_semi-structured,bridges_gemma_structured,bridges_gemma_unstructured,bridges_llama_semi-structured,bridges_llama_structured,bridges_llama_unstructured,bridges_qwen_semi-structured,bridges_qwen_structured,bridges_qwen_unstructured
0,0,1529451,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1600006,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1593387,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1592477,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1590316,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_CVs = pd.read_excel("../outputs/clean_outputs/full_cv_triples.xlsx")
df_CVs.head()

,Unnamed: 0,id,CV_triples
0,0,ebf358ed252344af8d1dd3f4c8516cbf,[('candidate_ebf358ed252344af8d1dd3f4c8516cbf'...
1,1,6c6e7a728dcc4ade98e1367f4efe6fb7,[('candidate_6c6e7a728dcc4ade98e1367f4efe6fb7'...
2,2,b1afb88436354245a405bcf36f4030c8,[('candidate_b1afb88436354245a405bcf36f4030c8'...
3,3,e9e192123ed44166993497619c499bd5,[('candidate_e9e192123ed44166993497619c499bd5'...
4,4,46353e9962684179846f77175feebef9,[('candidate_46353e9962684179846f77175feebef9'...


In [7]:
def extract_triples(input_string):
    """
    Extracts only 3-element tuples, accepting single or double quotes.
    """

    input_string = str(input_string)
    # Standardize stars to double quotes if that's your convention
    processed_string = input_string.replace('*', '"')

    # This pattern is much faster because it explicitly forbids internal quotes
    # and uses non-capturing groups for the quotes themselves.
    # 1. [^"']+ matches one or more characters that are NOT " or '
    # 2. We use | to handle either "..." or '...' explicitly
    
    element = r'(?:"[^"]*"|\'[^\']*\')'
    pattern = rf'\(\s*({element})\s*,\s*({element})\s*,\s*({element})\s*\)'

    matches = re.findall(pattern, processed_string, re.DOTALL)
    
    extracted_data = []
    for m in matches:
        # Clean quotes and whitespace efficiently
        trio = tuple(s.strip(" '\"") for s in m)
        if len(trio) == 3:
            extracted_data.append(trio)
            
    return list(set(extracted_data))

In [8]:
def clean_entity(entity):
    if not isinstance(entity, str):
        return str(entity)
    
    return entity.replace("_", " ").strip().lower()

def extract_entities(triples_input):
    # (The one that was working for individual models)
    try:
        raw_triples = extract_triples(triples_input)
    except Exception:
        print(e)
        return []

    if not raw_triples:
        return []

    # Delete duplicates
    return list(set(raw_triples))

def extract_entities_from_list(triples_list):
    """
    Now takes the list of tuples directly and returns unique subjects/objects.
    """
    unique_in_row = set()
    for t in triples_list:
        try:
            if len(t) >= 3:
                unique_in_row.add(clean_entity(t[0]))
                unique_in_row.add(clean_entity(t[2]))
        except (IndexError, TypeError):
            continue
    return list(unique_in_row)

def process_and_update(series):
    """Helper to process a column and dump directly into the set"""
    for row_triples in series:
        entities = extract_entities(row_triples)
        unique_entities.update(entities)


# 1. Helper to save unique entities per model
def save_model_entities(model_name, dataframe_list, filenames):
    unique_for_model = set()
    print(f"\n>>> Harvesting entities for {model_name}...")
    
    for df, col_suffix in dataframe_list:
        # Check all possible columns for this model/prompt combo
        for col in df.columns:
            if (model_name in col) or (col == "CV_triples"):
                print(f"  Processing and overwriting {col}...")
                
                # We use tqdm with pandas by using progress_apply
                tqdm.pandas(desc=f"Extracting {col}")
                df[col] = df[col].progress_apply(extract_triples)

                # 2. Extract unique entities for harvesting from the now-cleaned column
                for triples_list in df[col]:
                    entities = extract_entities_from_list(triples_list)
                    unique_for_model.update(entities)
        
    # Save this model's unique set to a temp file
    temp_file = f"temp_{model_name}.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        for e in unique_for_model:
            f.write(e + "\n")
    
    print(f"Done. Saved {len(unique_for_model)} unique entities to {temp_file}")
    
    del unique_for_model
    gc.collect()

# 2. Run the harvesting
model_data = [(triples, ""), (df_CVs, ""), (df_isco, ""), (df_bridges, ""), (df_isco_cv, "cv")]

for m in ["llama", "qwen", "gemma"]:
    if os.path.exists(f"temp_{m}.txt"):
        print(f">>> Found checkpoint for {m}. Skipping extraction.")
        continue
    
    save_model_entities(m, model_data, [])

# 3. Final Aggregation
print("\n>>> Merging all checkpoints...")
final_unique_entities = set()
for model in ["qwen", "gemma", "llama"]:
    file = f"temp_{model}.txt"
    if os.path.exists(file):
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                final_unique_entities.add(line.strip())

entity_list = list(final_unique_entities)
print(f"Final Unique Entity Count: {len(entity_list)}")

>>> Found checkpoint for llama. Skipping extraction.
>>> Found checkpoint for qwen. Skipping extraction.
>>> Found checkpoint for gemma. Skipping extraction.

>>> Merging all checkpoints...
Final Unique Entity Count: 931844


In [9]:
clean_entity("it-direktør")

'it-direktør'

In [10]:
"flittig pdagog" in entity_list

True

In [11]:
# 1. Setup paths and device
model_path = "opus-mt-da-en-ct2"
device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"

print("Loading tokenizer and CTranslate2 engine...")
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-da-en")
translator = ctranslate2.Translator(model_path, device=device)

def translate_entities(entities, batch_size=1024):
    """
    Translates 700k+ entities efficiently.
    """
    # Filter and unique-ify
    unique_entities = list(set(str(e).strip() for e in entities if e))
    print(f"Translating {len(unique_entities)} unique entities...")

    translated_map = {}
    
    # Process in batches
    for i in tqdm(range(0, len(unique_entities), batch_size)):
        batch = unique_entities[i : i + batch_size]
        
        # This prevents the 'Position Encoding' crash by cutting off LLM hallucinations
        source = [
            tokenizer.convert_ids_to_tokens(
                tokenizer.encode(e, truncation=True, max_length=100)
            ) for e in batch
        ]
        
        try:
            # beam_size=1 is "Greedy Search" - the fastest possible mode
            results = translator.translate_batch(
                source, 
                beam_size=1,
                max_batch_size=batch_size,
                batch_type="examples",
                replace_unknowns=True
            )
            
            for original, res in zip(batch, results):
                tokens = res.hypotheses[0]
                translated_map[original] = tokenizer.decode(
                    tokenizer.convert_tokens_to_ids(tokens), 
                    skip_special_tokens=True
                )
        except Exception as e:
            print(f"Error in batch starting at index {i}: {e}")
            continue

    # Map results back to the original input list to maintain order
    return {e: translated_map.get(str(e), e) for e in entities}

da_to_en_map = translate_entities(entity_list)

Loading tokenizer and CTranslate2 engine...
Translating 931843 unique entities...


  0%|          | 0/911 [00:00<?, ?it/s]

In [42]:
# 1. Setup Model
word_embedding_model = models.Transformer('jjzha/dajobbert-base-uncased')

# Mean pooling
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(), 
    pooling_mode='mean'
)

# Compile into a SentenceTransformer
model = SentenceTransformer(modules=[word_embedding_model, pooling_model]).to(device)

Loading base dajobbert model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
class TensorEncoder(json.JSONEncoder):
    def default(self, obj):
        if torch.is_tensor(obj):
            return obj.tolist()
        return super().default(obj)

In [44]:
import pickle # JSON is very slow for large numerical data

if os.path.exists("entity_embeddings.pkl"):
    print("Loading cached embeddings...")
    with open("entity_embeddings.pkl", "rb") as f:
        entity_embs = pickle.load(f)
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer('dajobbert-kg-specialized').to(device)
    
    # Pre-clean the list and ensure uniqueness
    # 700k strings is a lot; processing unique items only is mandatory
    unique_entities = [str(e).replace("_", " ") for e in entity_list if not (isinstance(e, float) and np.isnan(e))]
    unique_entities = list(set(unique_entities)) 

    print(f"Encoding {len(unique_entities)} entities on {device}...")

    embeddings = model.encode(
        unique_entities, 
        batch_size=128, 
        show_progress_bar=True, 
        convert_to_numpy=True # Numpy is better for saving/FAISS
    )

    # Rebuild the dictionary mapping
    entity_embs = dict(zip(unique_entities, embeddings))

    # Save as Pickle (much faster and smaller than JSON for 700k vectors)
    print("Saving embeddings...")
    with open("entity_embeddings.pkl", "wb") as f:
        pickle.dump(entity_embs, f)

# Update entity_list to reflect the processed entities
# entity_list = list(entity_embs.keys())

Loading cached embeddings...


In [45]:
ordered_keys = list(entity_embs.keys())
# entity_list = ordered_keys
ordered_embeddings = [torch.Tensor(entity_embs[key]) for key in ordered_keys]

embeddings = torch.stack(ordered_embeddings).to("cpu")
embeddings_array = F.normalize(embeddings, p=2, dim=1).to(torch.float32).numpy()
# similarity_matrix = cosine_similarity(embeddings_array.numpy(), embeddings_array.numpy())

In [46]:
embeddings_array.shape

(695764, 768)

In [47]:
def resolve_entities(entity_list, da_to_en_map, en_to_emb_map, string_threshold=0.80, semantic_threshold=0.98):
    """
    da_to_en_map: Dictionary { "Danish": "English" }
    en_to_emb_map: Dictionary { "English": [vector] }
    """
    
    unique_en_entities = list(en_to_emb_map.keys())
    G = nx.Graph()
    G.add_nodes_from(unique_en_entities)
    
    # ---------------------------------------------------------
    # 1. STRING SIMILARITY (HNSW Optimized)
    # ---------------------------------------------------------
    print(">>> Phase 1/3: String Similarity (TF-IDF + HNSW)...")
    # Reduced max_features to 1024 to speed up vectorization
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 3), max_features=1024)
    tfidf_matrix = vectorizer.fit_transform(unique_en_entities).toarray().astype('float32')
    faiss.normalize_L2(tfidf_matrix)

    # HNSW Index: M=32 is a good balance between speed and accuracy
    d_str = tfidf_matrix.shape[1]
    index_str = faiss.IndexHNSWFlat(d_str, 32)
    index_str.hnsw.efConstruction = 40 # Accuracy during building
    index_str.add(tfidf_matrix)
    
    # Search for top 5 neighbors
    distances, indices = index_str.search(tfidf_matrix, k=5)
    for i in range(len(unique_en_entities)):
        for dist, idx in zip(distances[i], indices[i]):
            if idx != -1 and idx != i and dist >= string_threshold:
                G.add_edge(unique_en_entities[i], unique_en_entities[idx])

    del tfidf_matrix # Free up RAM

    # ---------------------------------------------------------
    # 2. SEMANTIC SIMILARITY (HNSW Optimized)
    # ---------------------------------------------------------
    print(">>> Phase 2/3: Semantic Similarity (BERT + HNSW)...")
    embeddings_matrix = np.stack(list(en_to_emb_map.values())).astype('float32')
    faiss.normalize_L2(embeddings_matrix)
    
    d_sem = embeddings_matrix.shape[1]
    # HNSW is incredibly fast for BERT vectors
    index_sem = faiss.IndexHNSWFlat(d_sem, 32)
    index_sem.add(embeddings_matrix)
    
    distances, indices = index_sem.search(embeddings_matrix, k=5)
    for i in range(len(unique_en_entities)):
        for dist, idx in zip(distances[i], indices[i]):
            if idx != -1 and idx != i and dist >= semantic_threshold:
                G.add_edge(unique_en_entities[i], unique_en_entities[idx])

    # ---------------------------------------------------------
    # 3. RESOLVE CLUSTERS
    # ---------------------------------------------------------
    print(">>> Phase 3/3: Resolving Clusters...")
    
    # If a node has more than 10 connections, it's likely a 'bridge' that shouldn't exist
    hub_threshold = 10
    to_remove = [node for node, degree in dict(G.degree()).items() if degree > hub_threshold]
    print(f"Removing {len(to_remove)} hub entities that are causing over-merging...")
    G.remove_nodes_from(to_remove)

    # This identifies "tight" groups even if they have a stray edge to another group
    print("Partitioning clusters using Louvain algorithm...")
    partition = community_louvain.best_partition(G)
    
    # Group entities by their partition ID
    clusters = {}
    for entity, cluster_id in partition.items():
        if cluster_id not in clusters:
            clusters[cluster_id] = []
        clusters[cluster_id].append(entity)

    entity_counts = Counter(entity_list)
        
    coalesce_map = {}
    for cluster_id, members in tqdm(clusters.items(), desc="Standardizing Names"):
        # Canonical is the most common node in the cluster
        canonical = max(members, key=lambda x: entity_counts.get(x, 0))
        for entity in members:
            coalesce_map[entity] = canonical
            
    # Add back the hubs we removed as their own unique entities (so they don't disappear)
    for hub in to_remove:
        coalesce_map[hub] = hub

    # Extract the canonical cluster names (the correctly spelled "hubs")
    canonical_targets = list(set(coalesce_map.values()))

    unique_inputs = set(entity_list) 
    
    print(">>> Starting optimized fuzzy matching...")

    # 1. ONLY target the valid cluster hubs (entities that have >1 item mapped to them)
    target_counts = Counter(coalesce_map.values())
    canonical_targets = [target for target, count in target_counts.items() if count > 1]
    
    # 2. Get the list of isolated singletons
    singletons = [e for e in unique_inputs if coalesce_map.get(e, e) == e]
    
    print(f"Comparing {len(singletons)} singletons against {len(canonical_targets)} valid hubs...")
    
    # 3. Use C++ multithreaded matrix comparison (workers=-1 uses all CPU cores)
    # We use np.uint8 because fuzz.ratio is always between 0-100, saving massive amounts of RAM
    scores = process.cdist(
        singletons, 
        canonical_targets, 
        scorer=fuzz.ratio, 
        workers=-1, 
        dtype=np.uint8 
    )
    
    # 4. Vectorized extraction of the best matches
    max_scores = np.max(scores, axis=1)
    best_indices = np.argmax(scores, axis=1)
    
    # 5. Update the map for anything scoring 88 or higher
    matches_found = 0
    for i, singleton in enumerate(tqdm(singletons)):
        if max_scores[i] >= 88:
            coalesce_map[singleton] = canonical_targets[best_indices[i]]
            matches_found += 1
        
    print(">>> Creating Universal Mapping (Mixed DA/EN)...")
    
    final_master_map = {}
    
    for entity in unique_inputs:
        # This groups "flittig pdagog" -> "flittig pædagog"
        canonical_original = coalesce_map.get(entity, entity)
        
        # This safely translates "flittig pædagog" -> "diligent educator"
        canonical_en = da_to_en_map.get(canonical_original, canonical_original)
        
        final_master_map[entity] = canonical_en      

    return clusters, final_master_map

unique_entities = list(set(entity_list))
clusters, coalesce_map = resolve_entities(entity_list, da_to_en_map, entity_embs)

>>> Phase 1/3: String Similarity (TF-IDF + HNSW)...
>>> Phase 2/3: Semantic Similarity (BERT + HNSW)...
>>> Phase 3/3: Resolving Clusters...
Removing 1270 hub entities that are causing over-merging...
Partitioning clusters using Louvain algorithm...


Standardizing Names:   0%|          | 0/664720 [00:00<?, ?it/s]

>>> Starting optimized fuzzy matching...
Comparing 915003 singletons against 5500 valid hubs...


  0%|          | 0/915003 [00:00<?, ?it/s]

>>> Creating Universal Mapping (Mixed DA/EN)...


In [48]:
def generate_english_similarity_triples(entity_embs, coalesce_map, da_to_en_map, similarity_threshold=0.88, k_neighbors=10):
    # 1. Get Canonical Native (Danish) Entities
    # coalesce_map.values() contains the correctly spelled Danish hubs
    canonical_native_entities = list(set(coalesce_map.values()))
    print(f">>> Processing {len(canonical_native_entities)} native canonical entities...")
    
    valid_embs = []
    valid_keys_native = []
    
    for key in canonical_native_entities:
        # FILTER: Ignore sentence fragments and garbage LLM extractions
        # Only keep entities that are 1 to 4 words long
        word_count = len(str(key).split())
        
        if key in entity_embs and 0 < word_count <= 4:
            valid_keys_native.append(key)
            valid_embs.append(entity_embs[key])
            
    # 2. Build FAISS Index and Search on NATIVE Embeddings
    embeddings_matrix = np.stack(valid_embs).astype('float32')
    faiss.normalize_L2(embeddings_matrix)
    
    d_sem = embeddings_matrix.shape[1]
    index_sem = faiss.IndexHNSWFlat(d_sem, 32)
    index_sem.add(embeddings_matrix)
    
    distances, indices = index_sem.search(embeddings_matrix, k=k_neighbors)
    
    # 3. Extract, Translate, and Clean
    similarity_triples = []
    for i in tqdm(range(len(valid_keys_native)), desc="Building English Edges"):
        native_A = valid_keys_native[i]
        
        # Translate Node A safely
        english_A = da_to_en_map.get(native_A, native_A)
        
        for dist, idx in zip(distances[i], indices[i]):
            if idx != -1 and idx != i and dist >= similarity_threshold:
                native_B = valid_keys_native[idx]
                
                # Translate Node B safely
                english_B = da_to_en_map.get(native_B, native_B)
                
                # Prevent English self-loops (e.g. if "udvikler" and "programmør" 
                # were both translated to "developer" by the MT model)
                if english_A != english_B:
                    similarity_triples.append((english_A, "SIMILAR_TO", english_B))
                    
    # Deduplicate
    similarity_triples = list(set(similarity_triples))
    print(f"Generated {len(similarity_triples)} English SIMILAR_TO triples.")
    
    return similarity_triples

# Run the generator
sim_edges = generate_english_similarity_triples(entity_embs, coalesce_map, da_to_en_map)

# Save to Excel
df_sim = pd.DataFrame({"similarity_triples": sim_edges})
df_sim.to_excel("../outputs/final_outputs/similarity_triples.xlsx", index=False)
df_sim.head()

>>> Processing 839704 native canonical entities...


Building English Edges:   0%|          | 0/341017 [00:00<?, ?it/s]

Generated 1115 English SIMILAR_TO triples.


,similarity_triples
0,"(institution, SIMILAR_TO, easier reporting;)"
1,"(discount, SIMILAR_TO, germany is)"
2,"(champ, SIMILAR_TO, rights checks;)"
3,"(ant, SIMILAR_TO, 4209)"
4,"(3 19, SIMILAR_TO, language skills: Galician)"


In [49]:
# Check 1: Did we accidentally filter out the correct spelling?
target_counts = Counter(coalesce_map.values())
print("Count of correct spelling:", target_counts.get("flittig pædagog", 0))

# Check 2: What did the translation model actually do?
print("Translation of typo:", da_to_en_map.get("flittig pdagog"))
print("Translation of correct spelling:", da_to_en_map.get("flittig pædagog"))

Count of correct spelling: 0
Translation of typo: diligent pdagog
Translation of correct spelling: None


In [50]:
coalesce_map["programmør"]

'programmer'

In [51]:
coalesce_map["flittig pdagog"]

'diligent pdagog'

In [52]:
def check_cluster_stats(clusters):
    sizes = [len(c) for c in clusters.values()]
    size_counts = Counter(sizes)
    
    print("--- Cluster Statistics ---")
    print(f"Total Clusters: {len(clusters)}")
    print(f"Singletons (Unique Entities): {size_counts[1]}")
    print(f"Resolved (Clusters with >1 member): {len(clusters) - size_counts[1]}")
    
    print("\n--- Top 5 Largest Clusters ---")
    sorted_sizes = sorted(sizes, reverse=True)
    for i, s in enumerate(sorted_sizes[:5]):
        print(f"Cluster {i+1}: {s} entities merged")
        
    if sorted_sizes[0] > 100:
        print("\nWARNING: Large cluster detected. You might need a higher threshold or community detection.")

# Run the check
check_cluster_stats(clusters)

--- Cluster Statistics ---
Total Clusters: 664720
Singletons (Unique Entities): 659220
Resolved (Clusters with >1 member): 5500

--- Top 5 Largest Clusters ---
Cluster 1: 417 entities merged
Cluster 2: 343 entities merged
Cluster 3: 323 entities merged
Cluster 4: 317 entities merged
Cluster 5: 253 entities merged



In [53]:
def map_list_of_triples_to_canonical(list_of_triples, canonical_map):
    """
    Iterates over a list of triples and updates the first (subject) 
    and third (object) elements of each triple using the canonical map.

    :param list_of_triples: A list where each item is a triple (a, b, c).
    :param canonical_map: The dictionary mapping original strings to canonical strings.
    :return: A new list of triples with canonicalized entities.
    """
    canonicalized_list = []

    for triple in extract_triples(list_of_triples):

        triple = triple
        
        if len(triple) != 3:
            # Handle malformed triples if necessary, or skip them
            canonicalized_list.append(triple)
            continue
            
        a, b, c = triple

        # Check and replace 'a' (subject)
        # .get(key, default) returns the original 'a' if not found
        new_a = canonical_map.get(a, a) 

        # Check and replace 'c' (object)
        new_c = canonical_map.get(c, c) 

        # Append the new, canonicalized triple
        canonicalized_list.append((new_a, b, new_c))
        
    return canonicalized_list

In [54]:
model_data = [(triples, "triples"), (df_CVs, "cv"), (df_isco, "triples_ISCO_esco"),
              (df_bridges, "bridges"), (df_isco_cv, "cv_esco")]

for df, name in model_data:
    # Check all possible columns for this model/prompt combo
    for col in df.columns:
        if ("qwen" in col) or ("gemma" in col) or ("llama" in col) or (col == "CV_triples"):      
            print(name, col)
            df[col] = df[col].apply(
                lambda list_of_triples: map_list_of_triples_to_canonical(list_of_triples, coalesce_map)
            )

    df.to_excel(f"../outputs/final_outputs/{name}.xlsx")

triples triples_qwen_structured
triples triples_qwen_semi-structured
triples triples_qwen_unstructured
triples triples_gemma_structured
triples triples_gemma_semi-structured
triples triples_gemma_unstructured
triples triples_llama_structured
triples triples_llama_semi-structured
triples triples_llama_unstructured
cv CV_triples
triples_ISCO_esco ISCO cv qwen
triples_ISCO_esco ESCO cv qwen
triples_ISCO_esco ISCO gemma semi-structured
triples_ISCO_esco ESCO gemma semi-structured
triples_ISCO_esco ISCO gemma structured
triples_ISCO_esco ESCO gemma structured
triples_ISCO_esco ISCO gemma unstructured
triples_ISCO_esco ESCO gemma unstructured
triples_ISCO_esco ISCO llama semi-structured
triples_ISCO_esco ESCO llama semi-structured
triples_ISCO_esco ISCO llama structured
triples_ISCO_esco ESCO llama structured
triples_ISCO_esco ISCO llama unstructured
triples_ISCO_esco ESCO llama unstructured
triples_ISCO_esco ISCO qwen semi-structured
triples_ISCO_esco ESCO qwen semi-structured
triples_ISCO_

In [35]:
def map_json_entities_to_canonical(json_list, canonical_map, keys_to_map=None):
    """
    Updates entities within lists under specified keys in a list of dictionaries (JSON format)
    to their canonical forms.

    :param json_list: The list of dictionaries (your JSON data).
    :param canonical_map: The dictionary mapping original strings to canonical strings.
    :param keys_to_map: The keys whose values (lists of entities) should be mapped.
                        Defaults to ['job_history', 'jobtitles', 'keywords'].
    :return: A new list of dictionaries with canonicalized entity lists.
    """
    if keys_to_map is None:
        keys_to_map = ['job_history', 'jobtitles', 'keywords']
        
    canonicalized_json = []

    for entity_dict in json_list:
        new_dict = {}
        
        # Iterate over all keys in the dictionary
        for key, value in entity_dict.items():
            
            # Check if the key is one we need to map AND if the value is a list
            if key in keys_to_map and isinstance(value, list):
                
                canonicalized_list = []
                # Iterate through every entity in the list value
                for entity in value:
                    # Replace the entity with its canonical form if found in the map
                    # Otherwise, keep the original entity string
                    canonical_entity = canonical_map.get(str(entity).lower(), str(entity).lower())
                    canonicalized_list.append(canonical_entity)
                
                new_dict[key] = list(set(canonicalized_list))
            
            # If the key is not one to map, or the value is not a list, copy it as is
            else:
                new_dict[key] = value
                
        canonicalized_json.append(new_dict)
        
    return canonicalized_json

In [38]:
coalesced_data = map_json_entities_to_canonical(data, coalesce_map)

NameError: name 'data' is not defined

In [39]:
coalesced_data[:10]

NameError: name 'coalesced_data' is not defined

In [ ]:
with open("../outputs/final_outputs/anon_cvs_coalesced.json", 'w') as json_file:
        # json.dump() writes the dictionary to the file object
        json.dump(coalesced_data, json_file, indent=4)